In [ ]:
# Public-repository path setup.
# Run from anywhere inside the repository, or set CEFTAZIDIME_PROJECT_ROOT.
import os
from pathlib import Path

def _repo_root():
    env = os.environ.get("CEFTAZIDIME_PROJECT_ROOT")
    if env:
        return Path(env).expanduser().resolve()
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "README.md").exists() and (candidate / "03_Notebooks").exists():
            return candidate
    return here

def _previous_project_root(project_root):
    env = os.environ.get("GENOME_MIC_AMR_PROJECT_ROOT")
    if env:
        return Path(env).expanduser().resolve()
    return (project_root / "external" / "Genome_MIC_AMR_Emergence").resolve()

PROJECT_ROOT = _repo_root()

#@title Cell 18.1 - Overview, paths, and fixed analysis definition
# Purpose:
# Test whether all 1,287,844 variable chromosomal unitigs together reconstruct
# a significant association with continuous log2 ceftazidime MIC in the
# 176 blaTEM-1-only pathogens.
#
# Primary analysis:
# 1. Build a 176 x 176 whole-sequence similarity matrix K_unitig from all
#    variable unitig presence/absence states.
# 2. Use the same frequency-centred similarity formula used for the previous
#    SNP-based chromosomal similarity matrix K:
#
#       K_ij = sum_m[(X_im - p_m)(X_jm - p_m)]
#              / sum_m[p_m(1 - p_m)]
#
#    where X_im is 0/1 unitig presence and p_m is its frequency.
# 3. Fit the intercept-only mixed model by REML:
#
#       y = beta_0 + u + e
#       u ~ N(0, sigma_g^2 K_unitig)
#       e ~ N(0, sigma_e^2 I)
#
# 4. Calculate the whole-sequence variance fraction:
#
#       f = sigma_g^2 / (sigma_g^2 + sigma_e^2)
#
# 5. Test significance with 1,000 MIC permutations while K_unitig is fixed,
#    matching the earlier chromosomal-background analysis.
#
# This notebook tests the collective sequence association only.
# It does not remove unitigs or identify a causal combination.

from pathlib import Path
import json
import re
import time

import numpy as np
import pandas as pd
from scipy import sparse, optimize, stats
from IPython.display import display
PROJECT_ROOT = _repo_root()
NOTEBOOK_DIR = PROJECT_ROOT / '03_Notebooks' / '04_Genome_Comparison'
RESULTS_TABLE_DIR = PROJECT_ROOT / '05_Results' / 'Tables'

UNITIG_DIR = PROJECT_ROOT / '04_Intermediate' / '10_Whole_Chromosome_Unitigs'
UNITIG_MATRIX = UNITIG_DIR / '10_variable_unitig_matrix_176xM.npz'
UNITIG_SAMPLES = UNITIG_DIR / '10_unitig_sample_order.csv'
NB10_QC = RESULTS_TABLE_DIR / '10_unitig_representation_final_QC.csv'

OLD_K_FILE = (
    MYDRIVE
    / 'Genome_MIC_AMR_Emergence'
    / '04_Population_Structure'
    / 'Notebook04'
    / '04_genome_wide_relatedness_matrix.npz'
)

OLD_K_INDEX_FILE = (
    MYDRIVE
    / 'Genome_MIC_AMR_Emergence'
    / '02_Data_Preparation'
    / 'Notebook03'
    / '03_ceftazidime_pathogen_index.csv'
)

NB18_DIR = (
    PROJECT_ROOT
    / '04_Intermediate'
    / '18_Collective_Whole_Chromosome_Association'
)

NB18_DIR.mkdir(parents=True, exist_ok=True)

K_UNITIG_FILE = NB18_DIR / '18_whole_sequence_unitig_similarity_matrix.npz'
K_UNITIG_INDEX = NB18_DIR / '18_whole_sequence_unitig_similarity_index.csv'
OLD_K_REPRODUCTION = RESULTS_TABLE_DIR / '18_previous_SNP_K_reproduction.csv'
K_COMPARISON = RESULTS_TABLE_DIR / '18_unitig_K_vs_previous_SNP_K_comparison.csv'
VARIANCE_COMPONENTS = RESULTS_TABLE_DIR / '18_collective_unitig_variance_components.csv'
PERMUTATION_RESULTS = NB18_DIR / '18_collective_unitig_permutation_results.csv.gz'
FINAL_SUMMARY = RESULTS_TABLE_DIR / '18_collective_unitig_association_summary.csv'
FINAL_QC = RESULTS_TABLE_DIR / '18_collective_unitig_association_final_QC.csv'
COMPLETION_FILE = NB18_DIR / '18_COLLECTIVE_WHOLE_CHROMOSOME_ASSOCIATION_COMPLETE.json'

EXPECTED_PATHOGENS = 176
EXPECTED_UNITIGS = 1_287_844
N_PERMUTATIONS = 1000
PERMUTATION_SEED = 20260914

for path in [
    PROJECT_ROOT,
    NOTEBOOK_DIR,
    RESULTS_TABLE_DIR,
    UNITIG_MATRIX,
    UNITIG_SAMPLES,
    NB10_QC,
    OLD_K_FILE,
    OLD_K_INDEX_FILE,
]:
    assert path.exists(), f'Required input not found: {path}'

print('Notebook 18 - Collective Whole-Chromosome Sequence Association with Ceftazidime MIC')
print('Pathogens:', EXPECTED_PATHOGENS)
print('Variable chromosomal unitigs:', f'{EXPECTED_UNITIGS:,}')
print('Permutations:', N_PERMUTATIONS)
print('Outcome: continuous log2 ceftazidime MIC')
print('No unitig removal or ablation is performed in this notebook.')
print('\nTransition: Cell 18.2 will verify and align the 176 pathogens, phenotype, unitig matrix, and previous SNP-based K.')


In [ ]:
#@title Cell 18.2 - Verify and align phenotype, unitigs, and previous K
# Purpose:
# Confirm the accepted Notebook 10 representation, establish the exact
# 176-pathogen order, and align the previous SNP-based chromosomal K to it.

def normalized_column_name(name):
    return re.sub(r'[^a-z0-9]+', '', str(name).lower())

def find_biosample_column(df):
    accepted = {'biosample', 'biosampleid', 'ncbibiosample'}
    matches = [
        column for column in df.columns
        if normalized_column_name(column) in accepted
    ]
    if len(matches) != 1:
        raise ValueError(
            'Could not identify exactly one BioSample column.\n'
            f'Available columns: {list(df.columns)}\n'
            f'Matches: {matches}'
        )
    return matches[0]

nb10_qc = pd.read_csv(NB10_QC)
assert len(nb10_qc) == 1
assert bool(nb10_qc.loc[0, 'final_QC_pass'])
assert int(nb10_qc.loc[0, 'variable_unitigs_M']) == EXPECTED_UNITIGS

samples = pd.read_csv(UNITIG_SAMPLES)
required_sample_columns = {'sample_index', 'biosample', 'assembly_accession', 'log2_mic'}
missing_sample_columns = required_sample_columns - set(samples.columns)
assert not missing_sample_columns, (
    'Unitig sample-order file is missing required columns: '
    + ', '.join(sorted(missing_sample_columns))
)

samples = samples.sort_values('sample_index').reset_index(drop=True)
assert len(samples) == EXPECTED_PATHOGENS
assert samples['biosample'].nunique() == EXPECTED_PATHOGENS
assert np.array_equal(
    samples['sample_index'].to_numpy(dtype=int),
    np.arange(EXPECTED_PATHOGENS),
)

biosamples = samples['biosample'].astype(str).tolist()
y = samples['log2_mic'].to_numpy(dtype=float)
assert y.shape == (EXPECTED_PATHOGENS,)
assert np.isfinite(y).all()

unitig_matrix_shape = sparse.load_npz(UNITIG_MATRIX).shape
assert unitig_matrix_shape == (EXPECTED_PATHOGENS, EXPECTED_UNITIGS)

old_k_npz = np.load(OLD_K_FILE)
square_arrays = []
for key in old_k_npz.files:
    array = np.asarray(old_k_npz[key])
    if array.ndim == 2 and array.shape[0] == array.shape[1]:
        square_arrays.append((key, array.astype(float)))

if len(square_arrays) == 1:
    old_k_array_name, old_K_full = square_arrays[0]
elif len(square_arrays) > 1 and 'K' in old_k_npz.files:
    old_k_array_name = 'K'
    old_K_full = np.asarray(old_k_npz['K'], dtype=float)
else:
    raise ValueError(
        'Could not identify the previous square K matrix. Available arrays: '
        + str({key: np.asarray(old_k_npz[key]).shape for key in old_k_npz.files})
    )

old_k_index = pd.read_csv(OLD_K_INDEX_FILE)
old_k_biosample_column = find_biosample_column(old_k_index)
old_k_biosamples = old_k_index[old_k_biosample_column].astype(str).tolist()
assert len(old_k_biosamples) == old_K_full.shape[0]
assert len(set(old_k_biosamples)) == len(old_k_biosamples)

old_k_position = {biosample: index for index, biosample in enumerate(old_k_biosamples)}
missing_from_old_k = [biosample for biosample in biosamples if biosample not in old_k_position]
assert not missing_from_old_k, (
    'Pathogens missing from previous K:\n' + '\n'.join(missing_from_old_k[:20])
)

old_positions = [old_k_position[biosample] for biosample in biosamples]
old_K = old_K_full[np.ix_(old_positions, old_positions)]
old_K = (old_K + old_K.T) / 2.0

assert old_K.shape == (EXPECTED_PATHOGENS, EXPECTED_PATHOGENS)
assert np.isfinite(old_K).all()
old_k_min_eigenvalue = float(np.linalg.eigvalsh(old_K).min())
assert old_k_min_eigenvalue > -1e-6

print('Notebook 10 QC: PASS')
print('Pathogens aligned:', len(biosamples))
print('Unitig matrix shape:', unitig_matrix_shape)
print('Continuous log2 ceftazidime MIC range:', float(y.min()), 'to', float(y.max()))
print('Previous K array:', old_k_array_name)
print('Previous K aligned shape:', old_K.shape)
print('Previous K minimum eigenvalue:', old_k_min_eigenvalue)

display(samples[['sample_index', 'biosample', 'assembly_accession', 'log2_mic']].head())

print('\nCell 18.2 complete.')
print('Transition: Cell 18.3 will reproduce the previous SNP-based chromosomal-background variance result using the same 176 pathogens.')


In [ ]:
#@title Cell 18.3 - Reproduce the previous SNP-based chromosomal-background result
# Purpose:
# Validate the REML implementation by reproducing the established blaTEM-1-only
# chromosomal-background variance fraction from the previous SNP-based K.
#
# The earlier study reported a variance fraction of 0.611 for these 176
# pathogens. This cell should reproduce that value to rounding tolerance.

def prepare_kernel(K):
    K = np.asarray(K, dtype=float)
    K = (K + K.T) / 2.0

    eigenvalues, eigenvectors = np.linalg.eigh(K)
    minimum_eigenvalue = float(eigenvalues.min())

    if minimum_eigenvalue < -1e-6:
        raise ValueError(
            f'K is not positive semidefinite: minimum eigenvalue = {minimum_eigenvalue}'
        )

    eigenvalues = np.maximum(eigenvalues, 0.0)
    transformed_intercept = eigenvectors.T @ np.ones(K.shape[0], dtype=float)

    return {
        'K': K,
        'eigenvalues': eigenvalues,
        'eigenvectors': eigenvectors,
        'transformed_intercept': transformed_intercept,
        'minimum_eigenvalue': minimum_eigenvalue,
    }

def fit_null_reml_prepared(y, prepared):
    y = np.asarray(y, dtype=float).reshape(-1)
    eigenvalues = prepared['eigenvalues']
    eigenvectors = prepared['eigenvectors']
    transformed_intercept = prepared['transformed_intercept']
    transformed_y = eigenvectors.T @ y

    n = len(y)
    degrees_of_freedom = n - 1

    def evaluate_ratio(ratio):
        if ratio < 0:
            return None

        covariance_eigenvalues = 1.0 + ratio * eigenvalues
        if np.any(covariance_eigenvalues <= 0):
            return None

        inverse_weights = 1.0 / covariance_eigenvalues
        information = float(
            np.sum(
                transformed_intercept
                * transformed_intercept
                * inverse_weights
            )
        )
        if information <= 0:
            return None

        beta_0 = float(
            np.sum(
                transformed_intercept
                * transformed_y
                * inverse_weights
            )
            / information
        )

        transformed_residual = transformed_y - beta_0 * transformed_intercept
        residual_quadratic = float(
            np.sum(
                transformed_residual
                * transformed_residual
                * inverse_weights
            )
        )
        if residual_quadratic <= 0:
            return None

        sigma_e2 = residual_quadratic / degrees_of_freedom
        sigma_g2 = ratio * sigma_e2
        log_determinant = float(np.log(covariance_eigenvalues).sum())

        objective = 0.5 * (
            degrees_of_freedom * np.log(sigma_e2)
            + log_determinant
            + np.log(information)
        )

        return {
            'objective': float(objective),
            'sigma_g2_to_sigma_e2_ratio': float(ratio),
            'sigma_g2': float(sigma_g2),
            'sigma_e2': float(sigma_e2),
            'intercept': float(beta_0),
        }

    def objective_on_log_ratio(log_ratio):
        result = evaluate_ratio(np.exp(log_ratio))
        if result is None:
            return np.inf
        return result['objective']

    optimized = optimize.minimize_scalar(
        objective_on_log_ratio,
        bounds=(-12.0, 12.0),
        method='bounded',
        options={'xatol': 1e-8, 'maxiter': 500},
    )

    candidates = []
    zero_result = evaluate_ratio(0.0)
    if zero_result is not None:
        candidates.append(zero_result)

    if optimized.success:
        optimized_result = evaluate_ratio(float(np.exp(optimized.x)))
        if optimized_result is not None:
            candidates.append(optimized_result)

    high_result = evaluate_ratio(float(np.exp(12.0)))
    if high_result is not None:
        candidates.append(high_result)

    assert candidates, 'REML optimization failed.'
    return min(candidates, key=lambda item: item['objective'])

old_prepared = prepare_kernel(old_K)
old_fit = fit_null_reml_prepared(y, old_prepared)
old_variance_fraction = float(
    old_fit['sigma_g2'] / (old_fit['sigma_g2'] + old_fit['sigma_e2'])
)

old_reproduction = pd.DataFrame([
    {
        'pathogens': EXPECTED_PATHOGENS,
        'sigma_g2': old_fit['sigma_g2'],
        'sigma_e2': old_fit['sigma_e2'],
        'variance_fraction': old_variance_fraction,
        'previous_reported_variance_fraction': 0.611,
        'absolute_difference_from_reported': abs(old_variance_fraction - 0.611),
    }
])

old_reproduction.to_csv(OLD_K_REPRODUCTION, index=False)
display(old_reproduction)

assert abs(old_variance_fraction - 0.611) < 0.01, (
    'The REML implementation did not reproduce the previous '
    'blaTEM-1 variance fraction closely enough.'
)

print('Previous SNP-based K reproduction: PASS')
print('\nCell 18.3 complete.')
print('Transition: Cell 18.4 will construct K_unitig from all 1,287,844 variable chromosomal unitigs.')


In [ ]:
#@title Cell 18.4 - Construct the whole-sequence unitig similarity matrix
# Purpose:
# Build K_unitig from every one of the 1,287,844 variable unitigs.
#
# The construction mirrors the previous SNP-based K:
#
#   K_ij = sum_m[(X_im - p_m)(X_jm - p_m)]
#          / sum_m[p_m(1 - p_m)]
#
# No MIC information is used here.
# Identical presence/absence patterns remain represented by their original
# number of unitigs; this is the primary "all unitigs together" analysis.

construction_start = time.time()

X = sparse.load_npz(UNITIG_MATRIX).tocsc()
assert X.shape == (EXPECTED_PATHOGENS, EXPECTED_UNITIGS)
assert np.issubdtype(X.dtype, np.integer)

if X.nnz:
    assert X.data.min() >= 1
    assert X.data.max() <= 1

presence_counts = np.asarray(X.sum(axis=0)).ravel().astype(np.float64)
assert presence_counts.shape == (EXPECTED_UNITIGS,)
assert presence_counts.min() >= 1
assert presence_counts.max() <= (EXPECTED_PATHOGENS - 1)

p = presence_counts / EXPECTED_PATHOGENS
denominator = float(np.sum(p * (1.0 - p)))
assert denominator > 0

X_int = X.astype(np.int32)
XX = (X_int @ X_int.T).toarray().astype(np.float64)
del X_int

Xp = np.asarray(X @ p).reshape(-1).astype(np.float64)
p_squared_sum = float(p @ p)

numerator = (
    XX
    - Xp[:, None]
    - Xp[None, :]
    + p_squared_sum
)

K_unitig = numerator / denominator
K_unitig = (K_unitig + K_unitig.T) / 2.0

assert K_unitig.shape == (EXPECTED_PATHOGENS, EXPECTED_PATHOGENS)
assert np.isfinite(K_unitig).all()

symmetry_error = float(np.max(np.abs(K_unitig - K_unitig.T)))
unitig_eigenvalues = np.linalg.eigvalsh(K_unitig)
unitig_min_eigenvalue = float(unitig_eigenvalues.min())
mean_diagonal = float(np.mean(np.diag(K_unitig)))

assert symmetry_error < 1e-10
assert unitig_min_eigenvalue > -1e-6
assert abs(mean_diagonal - 1.0) < 1e-8, (
    'K_unitig mean diagonal should equal 1 under this normalization.'
)

np.savez_compressed(K_UNITIG_FILE, K_unitig=K_unitig)

samples[
    ['sample_index', 'biosample', 'assembly_accession', 'log2_mic']
].to_csv(K_UNITIG_INDEX, index=False)

construction_minutes = (time.time() - construction_start) / 60.0

print('K_unitig construction: PASS')
print('Variable unitigs included:', f'{EXPECTED_UNITIGS:,}')
print('K_unitig shape:', K_unitig.shape)
print('Mean diagonal:', mean_diagonal)
print('Minimum eigenvalue:', unitig_min_eigenvalue)
print('Construction time:', f'{construction_minutes:.1f} minutes')
print('Saved K_unitig:', K_UNITIG_FILE)

print('\nCell 18.4 complete.')
print('Transition: Cell 18.5 will compare K_unitig with the previous SNP-based K and estimate the collective whole-sequence variance fraction.')


In [ ]:
#@title Cell 18.5 - Estimate the collective whole-sequence variance fraction
# Purpose:
# Fit the same intercept-only REML model using K_unitig instead of the
# previous SNP-based K.
#
# This asks whether all variable chromosomal unitigs together explain
# structured variation in continuous log2 ceftazidime MIC.

upper_triangle = np.triu_indices(EXPECTED_PATHOGENS, k=1)

off_diagonal_correlation = float(
    stats.pearsonr(
        K_unitig[upper_triangle],
        old_K[upper_triangle],
    ).statistic
)

k_comparison = pd.DataFrame([
    {
        'pathogens': EXPECTED_PATHOGENS,
        'variable_unitigs': EXPECTED_UNITIGS,
        'unitig_K_mean_diagonal': mean_diagonal,
        'unitig_K_minimum_eigenvalue': unitig_min_eigenvalue,
        'previous_K_mean_diagonal': float(np.mean(np.diag(old_K))),
        'previous_K_minimum_eigenvalue': old_k_min_eigenvalue,
        'off_diagonal_Pearson_correlation': off_diagonal_correlation,
    }
])

k_comparison.to_csv(K_COMPARISON, index=False)

unitig_prepared = prepare_kernel(K_unitig)
unitig_fit = fit_null_reml_prepared(y, unitig_prepared)
unitig_variance_fraction = float(
    unitig_fit['sigma_g2'] / (unitig_fit['sigma_g2'] + unitig_fit['sigma_e2'])
)

variance_components = pd.DataFrame([
    {
        'pathogens': EXPECTED_PATHOGENS,
        'variable_unitigs': EXPECTED_UNITIGS,
        'sigma_g2': unitig_fit['sigma_g2'],
        'sigma_e2': unitig_fit['sigma_e2'],
        'sigma_g2_to_sigma_e2_ratio': unitig_fit['sigma_g2_to_sigma_e2_ratio'],
        'whole_sequence_variance_fraction': unitig_variance_fraction,
        'previous_SNP_K_variance_fraction_reproduced': old_variance_fraction,
        'unitig_K_vs_previous_K_off_diagonal_Pearson_r': off_diagonal_correlation,
    }
])

variance_components.to_csv(VARIANCE_COMPONENTS, index=False)

print('K comparison:')
display(k_comparison)

print('\nCollective whole-sequence variance components:')
display(variance_components)

print('\nObserved whole-sequence variance fraction:', unitig_variance_fraction)

print('\nCell 18.5 complete.')
print('Transition: Cell 18.6 will perform 1,000 MIC permutations using fixed K_unitig.')


In [ ]:
#@title Cell 18.6 - Permutation test for the collective whole-sequence association
# Purpose:
# Test whether the observed K_unitig variance fraction is greater than expected
# if MIC values were unrelated to whole-chromosome sequence similarity.
#
# The observed log2 MIC values are randomly reassigned among the 176 pathogens
# while K_unitig remains fixed. REML is refitted for each of 1,000 permutations.
#
# Empirical p = (1 + number of permuted fractions >= observed fraction)
#               / (1 + number of permutations)

rng = np.random.default_rng(PERMUTATION_SEED)

permuted_fractions = np.empty(N_PERMUTATIONS, dtype=np.float64)
permuted_sigma_g2 = np.empty(N_PERMUTATIONS, dtype=np.float64)
permuted_sigma_e2 = np.empty(N_PERMUTATIONS, dtype=np.float64)

permutation_start = time.time()

for permutation_index in range(N_PERMUTATIONS):
    y_permuted = rng.permutation(y)
    permuted_fit = fit_null_reml_prepared(y_permuted, unitig_prepared)

    sigma_g2_perm = float(permuted_fit['sigma_g2'])
    sigma_e2_perm = float(permuted_fit['sigma_e2'])
    fraction_perm = float(
        sigma_g2_perm / (sigma_g2_perm + sigma_e2_perm)
    )

    permuted_sigma_g2[permutation_index] = sigma_g2_perm
    permuted_sigma_e2[permutation_index] = sigma_e2_perm
    permuted_fractions[permutation_index] = fraction_perm

    if (permutation_index + 1) % 100 == 0:
        print(
            'Completed permutations:',
            permutation_index + 1,
            '/',
            N_PERMUTATIONS,
        )

number_equal_or_greater = int(
    np.sum(permuted_fractions >= unitig_variance_fraction)
)

empirical_p_value = float(
    (1 + number_equal_or_greater)
    / (N_PERMUTATIONS + 1)
)

permutation_table = pd.DataFrame(
    {
        'permutation': np.arange(1, N_PERMUTATIONS + 1, dtype=int),
        'sigma_g2': permuted_sigma_g2,
        'sigma_e2': permuted_sigma_e2,
        'variance_fraction': permuted_fractions,
    }
)

permutation_table.to_csv(
    PERMUTATION_RESULTS,
    index=False,
    compression='gzip',
)

permutation_minutes = (time.time() - permutation_start) / 60.0

print('\nPermutation test complete.')
print('Observed whole-sequence variance fraction:', unitig_variance_fraction)
print('Permuted fractions >= observed:', number_equal_or_greater)
print('Empirical p-value:', empirical_p_value)
print('Permutation median:', float(np.median(permuted_fractions)))
print('Permutation 95th percentile:', float(np.quantile(permuted_fractions, 0.95)))
print('Elapsed time:', f'{permutation_minutes:.1f} minutes')

print('\nCell 18.6 complete.')
print('Transition: Cell 18.7 will perform final QC and state whether the collective whole-sequence association was reconstructed.')


In [ ]:
#@title Cell 18.7 - Final QC and interpretation
# Purpose:
# Verify all outputs and state the stopping decision.
#
# If the empirical p-value is < 0.05, the whole-chromosome unitig
# representation has reconstructed a significant collective association
# with ceftazidime MIC. Ablation is a later analysis, not part of this notebook.

assert K_UNITIG_FILE.exists()
assert K_UNITIG_INDEX.exists()
assert OLD_K_REPRODUCTION.exists()
assert K_COMPARISON.exists()
assert VARIANCE_COMPONENTS.exists()
assert PERMUTATION_RESULTS.exists()

saved_k = np.load(K_UNITIG_FILE)['K_unitig']
assert saved_k.shape == (EXPECTED_PATHOGENS, EXPECTED_PATHOGENS)
assert np.allclose(saved_k, K_unitig, atol=1e-12, rtol=0.0)

saved_index = pd.read_csv(K_UNITIG_INDEX)
assert len(saved_index) == EXPECTED_PATHOGENS
assert saved_index['biosample'].astype(str).tolist() == biosamples

saved_permutations = pd.read_csv(PERMUTATION_RESULTS)
assert len(saved_permutations) == N_PERMUTATIONS
assert saved_permutations['variance_fraction'].between(0, 1).all()

association_reconstructed = bool(empirical_p_value < 0.05)

summary = pd.DataFrame([
    {
        'pathogens': EXPECTED_PATHOGENS,
        'variable_unitigs_used_together': EXPECTED_UNITIGS,
        'previous_SNP_K_variance_fraction_reproduced': old_variance_fraction,
        'whole_sequence_unitig_variance_fraction': unitig_variance_fraction,
        'permutations': N_PERMUTATIONS,
        'empirical_p_value': empirical_p_value,
        'permuted_fractions_equal_or_greater_than_observed': number_equal_or_greater,
        'unitig_K_vs_previous_K_off_diagonal_Pearson_r': off_diagonal_correlation,
        'significant_collective_association_reconstructed': association_reconstructed,
    }
])

summary.to_csv(FINAL_SUMMARY, index=False)

qc = pd.DataFrame([
    {
        'pathogens_aligned': len(biosamples),
        'variable_unitigs': EXPECTED_UNITIGS,
        'previous_K_reproduction_pass': abs(old_variance_fraction - 0.611) < 0.01,
        'unitig_K_symmetric': bool(symmetry_error < 1e-10),
        'unitig_K_positive_semidefinite': bool(unitig_min_eigenvalue > -1e-6),
        'unitig_K_mean_diagonal_one': bool(abs(mean_diagonal - 1.0) < 1e-8),
        'permutations_completed': len(saved_permutations),
        'final_QC_pass': True,
    }
])

qc.to_csv(FINAL_QC, index=False)

completion_payload = {
    'status': 'complete',
    'pathogens': EXPECTED_PATHOGENS,
    'variable_unitigs_used_together': EXPECTED_UNITIGS,
    'whole_sequence_unitig_variance_fraction': unitig_variance_fraction,
    'empirical_p_value': empirical_p_value,
    'permutations': N_PERMUTATIONS,
    'significant_collective_association_reconstructed': association_reconstructed,
    'final_QC_pass': True,
}

COMPLETION_FILE.write_text(
    json.dumps(completion_payload, indent=2),
    encoding='utf-8',
)

display(summary)
print('\nFinal QC: PASS')

if association_reconstructed:
    print(
        '\nStopping decision: a significant collective whole-sequence '
        'association with continuous log2 ceftazidime MIC was reconstructed '
        'from all 1,287,844 variable chromosomal unitigs together.'
    )
    print(
        'The next analysis can therefore examine which groups of unitigs '
        'can be removed without losing this association and which groups '
        'are required to retain it.'
    )
else:
    print(
        '\nStopping decision: the collective whole-sequence association '
        'was not significant under the 1,000-permutation test.'
    )
    print(
        'Do not begin ablation analysis unless this result is first reviewed.'
    )

print(
    '\nThis is a collective observational association. '
    'It does not identify a causal variant or causal combination.'
)
